In [ ]:
# Environment Sync, GPU Check & Auto-Dataset Download

import os, sys
from pathlib import Path
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

REPO_NAME = "food-classification-deep-learning"
BRANCH = "feature/kaveesha-efficientnetb0"

if 'google.colab' in sys.modules:
    print("[INFO] Running in Google Colab environment.")
    if not os.path.exists(f"/content/{REPO_NAME}"):
        !git clone -b {BRANCH} https://github.com/niRmana11/food-classification-deep-learning.git
        %cd /content/{REPO_NAME}
    else:
        %cd /content/{REPO_NAME}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}

    if f"/content/{REPO_NAME}" not in sys.path:
        sys.path.insert(0, f"/content/{REPO_NAME}")

# Locked experimental protocol: seed 42 (configs/config.yaml) for weight initialization
# and any framework-level randomness. The data loader seeds its own shuffling separately.
tf.keras.utils.set_random_seed(42)

# Verify GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[SUCCESS] GPU active: {gpus[0].name}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print("[WARNING] No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

# Auto-download dataset if missing (~5GB, skips if already extracted)
if not os.path.exists("data/raw/food-101/images"):
    !python -m src.data.download_food101

print(f"[INFO] Class folders found: {len(os.listdir('data/raw/food-101/images'))} (expect 101)")

In [ ]:
# Instantiate Standardized Data Loaders

from src.preprocessing.data_loader import get_food101_datasets
from src.models.efficientnetb0 import (
    build_efficientnetb0, unfreeze_top_blocks, compile_model, FINETUNE_BLOCKS
)

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

print("[INFO] Loading datasets via shared factory...")
train_ds, val_ds, test_ds = get_food101_datasets(
    data_dir="data/raw/food-101",
    splits_dir="data/splits",
    model_type="efficientnetb0",   # Pass-through: EfficientNet rescales internally
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

print("[SUCCESS] Train, Validation and Test pipelines loaded successfully.")

In [ ]:
# SMOKE TEST - validate the full training path on ~640 images before spending GPU hours.
# Set SMOKE_TEST = False and re-run this cell once it passes, then continue to Cell 3.

SMOKE_TEST = True

if SMOKE_TEST:
    print("[SMOKE] Building throwaway model (does not affect the real run)...")
    _m, _base = build_efficientnetb0()

    print("\n[SMOKE] Phase 1 - 1 epoch on 20 batches...")
    _m.fit(train_ds.take(20), validation_data=val_ds.take(5), epochs=1, verbose=1)

    print("\n[SMOKE] Phase 2 - unfreeze, RECOMPILE, 1 epoch...")
    _n = unfreeze_top_blocks(_base)
    compile_model(_m, learning_rate=1e-5)
    print(f"[SMOKE] Unfroze {_n} layers; trainable tensors = {len(_m.trainable_weights)}")
    _m.fit(train_ds.take(20), validation_data=val_ds.take(5), epochs=1, verbose=1)

    print("\n[SMOKE] Checking prediction / label alignment...")
    _p = _m.predict(test_ds.take(5), verbose=0)
    _y = np.concatenate([y.numpy() for _, y in test_ds.take(5)])
    assert len(_p) == len(_y), "Prediction/label length mismatch!"
    assert _p.shape[1] == 101, f"Expected 101 output classes, got {_p.shape[1]}"
    print(f"[SMOKE] OK - {len(_p)} predictions aligned with {len(_y)} labels.")

    del _m, _base
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)   # restore seed state for the real run
    print("\n[SMOKE] PASSED. Set SMOKE_TEST = False, re-run this cell, then go to Cell 3.")
else:
    print("[INFO] Smoke test skipped - proceeding to the full run.")

In [ ]:
# Model Instantiation & Callbacks (Phase 1: Frozen Feature Extraction)

RESULTS_DIR = Path("results/efficientnetb0")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model, base_model = build_efficientnetb0(
    input_shape=(224, 224, 3),
    num_classes=101,
    learning_rate=0.001,
    dropout_rate=0.3
)

# Save architecture summary to model_summary.txt (required artifact)
with open(RESULTS_DIR / "model_summary.txt", "w") as f:
    model.summary(print_fn=lambda x: f.write(x + "\n"))


def make_callbacks(phase: int, min_lr: float):
    """
    Standardized callbacks, parameterised by phase.

    min_lr must differ between phases: Phase 2 STARTS at 1e-5, so a 1e-5 floor would
    make ReduceLROnPlateau a no-op during fine-tuning. Phase 1 uses 1e-5, Phase 2 1e-7.
    """
    return [
        # Halt when val_loss stops improving; roll back to the best epoch's weights
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
        ),
        # Fires first (patience 2 < 5): one automatic LR rescue before EarlyStopping gives up
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2, min_lr=min_lr, verbose=1
        ),
        # Per-epoch history so a Colab disconnect still leaves usable training curves
        tf.keras.callbacks.CSVLogger(
            filename=str(RESULTS_DIR / f"history_phase{phase}.csv"), separator=",", append=False
        ),
        # Per-epoch best checkpoint so a disconnect costs one epoch, not the whole run
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(RESULTS_DIR / f"effnetb0_phase{phase}_best.weights.h5"),
            monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=0
        ),
    ]


trainable_p1 = int(sum(int(tf.size(w)) for w in model.trainable_weights))
print(f"[INFO] Total parameters      : {model.count_params():,}")
print(f"[INFO] Phase 1 trainable     : {trainable_p1:,} ({len(model.trainable_weights)} tensors)")
print("[INFO] Callbacks configured. Ready for Phase 1 training.")

In [ ]:
# Execute Phase 1 Training on Colab GPU (Frozen Backbone, lr=1e-3)

EPOCHS_PHASE1 = 12

print(f"[INFO] Starting EfficientNetB0 Phase 1 for up to {EPOCHS_PHASE1} epochs...")
print("[INFO] Backbone frozen - only the 131,941-parameter head is training.")
start_p1 = time.time()

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=make_callbacks(phase=1, min_lr=1e-5)
)

time_p1 = time.time() - start_p1
print(f"\n[SUCCESS] Phase 1 finished in {time_p1:.1f}s ({time_p1/60:.1f} min).")
print(f"[RESULT] Best Phase 1 val_accuracy : {max(history_p1.history['val_accuracy']):.4f}")
print(f"[RESULT] Best Phase 1 val_loss     : {min(history_p1.history['val_loss']):.4f}")
print(f"[RESULT] Epochs actually run       : {len(history_p1.history['loss'])}")